Ground Truth for test images were not provided. That is why we use the train dataset as the dataset on why we do a train-test-split. 

In [1]:
import shutil
from pathlib import Path

from tifffile import TiffFile
import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from tqdm import tqdm


In [2]:
ROOT=Path("../")

(ROOT / "local_dataset").mkdir(parents=True, exist_ok=True)
(ROOT / "local_dataset/csv_files").mkdir(parents=True, exist_ok=True)


if (ROOT / "local_dataset").exists() or (ROOT / "local_dataset/csv_files").exists():
    raise FileExistsError(
        "Les dossiers 'local_dataset' et/ou 'csv_files' sont déjà présents."
    )


FileExistsError: Les dossiers 'local_dataset' et/ou 'csv_files' sont déjà présents.

In [3]:
df = pd.read_csv("../train_labels_GY1QjFw.csv")

# Colonnes contenant les classes
class_cols = [
    c for c in df.columns
    if c not in ["sample_id", "no_data", "clouds"]
]

# Matrice binaire :
# 1 = classe présente dans l'image
# 0 = classe absente de l'image

Y = (df[class_cols].fillna(0).to_numpy() > 0).astype(int)


X = df["sample_id"]

# Première séparation : 80% train / 20% temporaire
msss_1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2, 
    random_state=42
)


train_idx, temp_idx = next(msss_1.split(X, Y))


X_temp = X.iloc[temp_idx]
Y_temp = Y[temp_idx]

msss_2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_rel_idx, test_rel_idx = next(
    msss_2.split(X_temp, Y_temp)
)

train_ids = X.iloc[train_idx]
val_ids = X_temp.iloc[val_rel_idx]
test_ids = X_temp.iloc[test_rel_idx]

In [4]:
CSV_DIR= Path(ROOT / "local_dataset/csv_files")

train_ids.to_csv(CSV_DIR/"train_images.csv", index=False)
val_ids.to_csv(CSV_DIR/"validation_images.csv", index=False)
test_ids.to_csv(CSV_DIR/"test_images.csv", index=False)

print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

Train: 14794
Validation: 1848
Test: 1849


In [5]:
train_df = df.iloc[train_idx].copy()
val_df = df.iloc[temp_idx[val_rel_idx]].copy()
test_df = df.iloc[temp_idx[test_rel_idx]].copy()


train_df.to_csv(CSV_DIR/"train_labels.csv", index=False)
val_df.to_csv(CSV_DIR/"validation_labels.csv", index=False)
test_df.to_csv(CSV_DIR/"test_labels.csv", index=False)

In [ ]:
LOCAL_DATASET_DIR=Path(ROOT / "local_dataset")
DATASET_FOLDER = Path("../dataset_UNZIPPED")


# Train
train_images_paths = [DATASET_FOLDER / f"train/images/{sid}.tif" for sid in train_df["sample_id"]]
train_masks_paths = [DATASET_FOLDER / f"train/masks/{sid}.tif" for sid in train_df["sample_id"]]

# Validation
val_images_paths = [DATASET_FOLDER / f"train/images/{sid}.tif" for sid in val_df["sample_id"]]
val_masks_paths = [DATASET_FOLDER / f"train/masks/{sid}.tif" for sid in val_df["sample_id"]]

# Test
test_images_paths = [DATASET_FOLDER / f"train/images/{sid}.tif" for sid in test_df["sample_id"]]
test_masks_paths = [DATASET_FOLDER / f"train/masks/{sid}.tif" for sid in test_df["sample_id"]]



In [ ]:
for split, images, masks in [
    ("train", train_images_paths, train_masks_paths),
    ("validation", val_images_paths, val_masks_paths),
    ("test", test_images_paths, test_masks_paths)
]:
    (LOCAL_DATASET_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (LOCAL_DATASET_DIR / split / "masks").mkdir(parents=True, exist_ok=True)

    for img in images:
        shutil.copy(img, LOCAL_DATASET_DIR / split / "images" / img.name)


    for mask in masks:
        shutil.copy(mask, LOCAL_DATASET_DIR / split / "masks" / mask.name)

In [8]:
assert set(train_ids).isdisjoint(set(val_ids))
assert set(train_ids).isdisjoint(set(test_ids))
assert set(val_ids).isdisjoint(set(test_ids))

print("Aucun chevauchement entre les trois ensembles.")

Aucun chevauchement entre les trois ensembles.


In [9]:
# Number of images per set

print(f"Train: {len(train_images_paths)}")
print(f"Val: {len(val_images_paths)}")
print(f"Test: {len(test_images_paths)}")

Train: 14794
Val: 1848
Test: 1849


In [10]:
# Max & min value pixel in dataset

global_min = float("inf")
global_max = float("-inf")

for path in train_images_paths:# + val_images_paths:
    with TiffFile(path) as t:
        img = t.asarray()

    global_min = min(global_min, img.min())
    global_max = max(global_max, img.max())

print("Minimum global :", global_min)
print("Maximum global :", global_max)

Minimum global : 1
Maximum global : 24356


In [12]:
LOCAL_DATASET_DIR = Path("../local_dataset")

# get all train images and masks
train_masks_paths = sorted(list(LOCAL_DATASET_DIR.glob('train/masks/*.tif')))
# get all test images
test_masks_paths = sorted(list(LOCAL_DATASET_DIR.glob('test/masks/*.tif')))
# get all validation images
val_masks_paths = sorted(list(LOCAL_DATASET_DIR.glob('validation/masks/*.tif')))


Train masks: 14794
Test masks: 1849
Validation masks: 1848
Current directory: c:\Users\ecwol\Documents\TPS\TPS 3A Cours\land-cover-predictive-modeling-from-satellite-images\framework
Dataset folder exists: True


In [17]:
# Compute total class distribution in train
def compute_class_counts(masks_paths, n_classes=10):
    """Return the cumulated class counts for all masks in the training set.
    Args:
        masks_paths (list[path-like]): paths to all masks 
    """
    cumcount = np.zeros((n_classes,), dtype=np.int64)
    for path in tqdm(masks_paths):
        with TiffFile(path) as tif:
            arr = tif.asarray()
            count = np.bincount(arr.ravel(), minlength=10)
            # hist, _ = np.histogram(arr, bins=n_classes, range=(0, 10)) # @todo: use bincount
            cumcount += count
    return cumcount
    


In [19]:
TRAIN_CLASS_COUNTS = compute_class_counts(train_masks_paths)
VAL_CLASS_COUNTS = compute_class_counts(val_masks_paths)
TEST_CLASS_COUNTS = compute_class_counts(test_masks_paths)


# Training set percentages
train_class_percentages = (
    TRAIN_CLASS_COUNTS/ TRAIN_CLASS_COUNTS.sum()
) * 100

test_class_dist = (
    TEST_CLASS_COUNTS / TEST_CLASS_COUNTS.sum()
) * 100

validation_class_dist = (
    VAL_CLASS_COUNTS / VAL_CLASS_COUNTS.sum()
) * 100


for i in range(10):
    print(
        f"Class {i}: "
        f"Train = {train_class_percentages[i]:.2f}%, "
        f"Validation = {validation_class_dist[i]:.2f}%, "
        f"Test = {test_class_dist[i]:.2f}%"
    )

  0%|          | 9/14794 [00:00<02:50, 86.86it/s]

100%|██████████| 1849/1849 [00:01<00:00, 1529.29it/s]

Class 0: Train = 0.00%, Validation = 0.00%, Test = 0.00%
Class 1: Train = 0.00%, Validation = 0.00%, Test = 0.00%
Class 2: Train = 5.03%, Validation = 5.17%, Test = 4.90%
Class 3: Train = 33.41%, Validation = 33.14%, Test = 33.62%
Class 4: Train = 22.94%, Validation = 22.72%, Test = 22.35%
Class 5: Train = 7.94%, Validation = 8.40%, Test = 7.71%
Class 6: Train = 27.43%, Validation = 27.67%, Test = 28.02%
Class 7: Train = 0.80%, Validation = 0.85%, Test = 0.82%
Class 8: Train = 0.00%, Validation = 0.00%, Test = 0.00%
Class 9: Train = 2.46%, Validation = 2.04%, Test = 2.57%


In [20]:
print(f"TRAIN_CLASS_COUNTS: {TRAIN_CLASS_COUNTS}")
print(f"VAL_CLASS_COUNTS: {VAL_CLASS_COUNTS}")
print(f"TEST_CLASS_COUNTS: {TEST_CLASS_COUNTS}")

TRAIN_CLASS_COUNTS: [        0     17441  48775625 323881666 222414642  76948228 265937117
   7749139       828  23814898]
VAL_CLASS_COUNTS: [       0     1249  6258733 40140226 27516024 10176292 33510997  1030736
       23  2476248]
TEST_CLASS_COUNTS: [       0     1953  5936667 40739089 27081711  9348526 33959019   995420
      220  3113459]
